In [28]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, Row
from pyspark.mllib.util import MLUtils
from pyspark.ml.feature import StringIndexer, PCA
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [29]:
# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("BreastCancerPrediction")

# Stop any existing SparkContext if it's running
if 'SpContext' in globals() and SpContext:
    SpContext.stop()
SpContext = SparkContext(conf = conf)

# Create a SparkSession
SpSession = SparkSession.builder.getOrCreate()

print("Spark Context and Session initialized.")

Spark Context and Session initialized.


In [30]:
!wget -O 8-breast-cancer.txt https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-breast-cancer.txt

--2026-04-07 01:47:49--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-breast-cancer.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 82965 (81K) [text/plain]
Saving to: ‘8-breast-cancer.txt’

8-breast-cancer.txt 100%[===================>]  81.02K  --.-KB/s    in 0.04s   

2026-04-07 01:47:50 (1.83 MB/s) - ‘8-breast-cancer.txt’ saved [82965/82965]



In [31]:
# Load the CSV file into an RDD
breastcancerData = SpContext.textFile("8-breast-cancer.txt")
breastcancerData.cache()

8-breast-cancer.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

In [32]:
def parse_libsvm_line_to_row(line):
    parts = line.strip().split(" ")
    label = float(parts[0])
    features_dict = {}
    for feature_str in parts[1:]:
        if ":" in feature_str:
            idx, val = feature_str.split(":")
            features_dict[int(idx)] = float(val)

    # Assuming 10 features, 1-indexed, and mapping them to named columns
    return Row(
        OUTCOME=label,
        RADIUS=features_dict.get(1, 0.0),
        TEXTURE=features_dict.get(2, 0.0),
        PERIMETER=features_dict.get(3, 0.0),
        AREA=features_dict.get(4, 0.0),
        SMOOTHNESS=features_dict.get(5, 0.0),
        COMPACTNESS=features_dict.get(6, 0.0),
        CONCAVITY=features_dict.get(7, 0.0),
        CONCAVE_POINTS=features_dict.get(8, 0.0),
        SYMMETRY=features_dict.get(9, 0.0),
        FRACTAL_DIMENSION=features_dict.get(10, 0.0),
    )

In [33]:
# Create a Data Frame from the data
breastcancerMap = breastcancerData.map(parse_libsvm_line_to_row)
breastcancerDf = SpSession.createDataFrame(breastcancerMap)
breastcancerDf.cache()

DataFrame[OUTCOME: double, RADIUS: double, TEXTURE: double, PERIMETER: double, AREA: double, SMOOTHNESS: double, COMPACTNESS: double, CONCAVITY: double, CONCAVE_POINTS: double, SYMMETRY: double, FRACTAL_DIMENSION: double]

In [34]:
# Add a numeric indexer for the label/target column
stringIndexer = StringIndexer(inputCol="OUTCOME", outputCol="IND_OUTCOME")
si_model = stringIndexer.fit(breastcancerDf)
breastcancerNormDf = si_model.transform(breastcancerDf)
breastcancerNormDf.cache()

DataFrame[OUTCOME: double, RADIUS: double, TEXTURE: double, PERIMETER: double, AREA: double, SMOOTHNESS: double, COMPACTNESS: double, CONCAVITY: double, CONCAVE_POINTS: double, SYMMETRY: double, FRACTAL_DIMENSION: double, IND_OUTCOME: double]

In [36]:
import pyspark.sql.functions as F

# Prepare data for ML (LabeledPoint format)
def transformToLabeledPoint(row) :
    lp = ( row["OUTCOME"], row["IND_OUTCOME"],
           Vectors.dense([row["RADIUS"],
                          row["TEXTURE"],
                          row["PERIMETER"],
                          row["AREA"],
                          row["SMOOTHNESS"],
                          row["COMPACTNESS"],
                          row["CONCAVITY"],
                          row["CONCAVE_POINTS"],
                          row["SYMMETRY"],
                          row["FRACTAL_DIMENSION"]]))
    return lp

breastcancerLp = breastcancerNormDf.rdd.map(transformToLabeledPoint)
breastcancerLpDf = SpSession.createDataFrame(breastcancerLp, ["outcome", "label", "features"])
breastcancerLpDf.cache()

print("Data loaded and prepared.")
breastcancerLpDf.show(5)

Data loaded and prepared.
+-------+-----+--------------------+
|outcome|label|            features|
+-------+-----+--------------------+
|    0.0|  0.0|[1000025.0,5.0,1....|
|    0.0|  0.0|[1002945.0,5.0,4....|
|    0.0|  0.0|[1015425.0,3.0,1....|
|    0.0|  0.0|[1016277.0,6.0,8....|
|    0.0|  0.0|[1017023.0,4.0,1....|
+-------+-----+--------------------+
only showing top 5 rows


In [37]:
print("=== Start building Random Forest model with original features ===")

# Split data into training and test sets
(trainingData_rf, testData_rf) = breastcancerLpDf.randomSplit([0.8, 0.2], seed=42)

# Create and train the RandomForestClassifier model
rfClassifier_original = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=10)
rfModel_original = rfClassifier_original.fit(trainingData_rf)

# Predict on the test data
predictions_original = rfModel_original.transform(testData_rf)
predictions_original.select("prediction","outcome","label").show(5)

# Evaluate model performance
evaluator_original = MulticlassClassificationEvaluator(predictionCol="prediction", \
                                                       labelCol="label", metricName="weightedPrecision")
precision_original = evaluator_original.evaluate(predictions_original)

print("\nRandom Forest Model (Original Features):")
print(f"  Precision = {precision_original}")
print(f"  Error Rate = {1 - precision_original}")

print("\n=== Confusion Matrix (Original Features) ===")
predictions_original.groupBy("label","prediction").count().show()

=== Start building Random Forest model with original features ===
+----------+-------+-----+
|prediction|outcome|label|
+----------+-------+-----+
|       0.0|    0.0|  0.0|
|       0.0|    0.0|  0.0|
|       0.0|    0.0|  0.0|
|       0.0|    0.0|  0.0|
|       0.0|    0.0|  0.0|
+----------+-------+-----+
only showing top 5 rows

Random Forest Model (Original Features):
  Precision = 0.9911025145067698
  Error Rate = 0.008897485493230217

=== Confusion Matrix (Original Features) ===
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|   46|
|  0.0|       1.0|    1|
|  0.0|       0.0|   63|
+-----+----------+-----+



In [38]:
print("=== Start applying PCA (K=4) ===")

# Apply PCA with k=4
pca = PCA(k=4, inputCol="features", outputCol="pcaFeatures")
pcaModel = pca.fit(breastcancerLpDf)
pcaResult = pcaModel.transform(breastcancerLpDf).select("label", "pcaFeatures")
pcaResult.show(truncate=False)

# Split the PCA-transformed data into training and test sets
(trainingData_pca, testData_pca) = pcaResult.randomSplit([0.8, 0.2], seed=42)

print("=== Start building Random Forest model with PCA features ===")

# Create and train the RandomForestClassifier model with PCA features
rfClassifier_pca = RandomForestClassifier(labelCol="label", featuresCol="pcaFeatures", numTrees=10)
rfModel_pca = rfClassifier_pca.fit(trainingData_pca)

# Predict on the test data
predictions_pca = rfModel_pca.transform(testData_pca)
predictions_pca.select("prediction","label").show(5)

# Evaluate model performance
evaluator_pca = MulticlassClassificationEvaluator(predictionCol="prediction", \
                                                  labelCol="label", metricName="weightedPrecision")
precision_pca = evaluator_pca.evaluate(predictions_pca)

print("\nRandom Forest Model (PCA Features):")
print(f"  Precision = {precision_pca}")
print(f"  Error Rate = {1 - precision_pca}")

print("\n=== Confusion Matrix (PCA Features) ===")
predictions_pca.groupBy("label","prediction").count().show()

=== Start applying PCA (K=4) ===
+-----+----------------------------------------------------------------------------------+
|label|pcaFeatures                                                                       |
+-----+----------------------------------------------------------------------------------+
|0.0  |[-1000024.9999955925,-5.727826532988675,0.03807326718641846,-3.162761116373462]   |
|0.0  |[-1002944.999986718,-15.02291570854395,4.809264765810976,-0.40975027079538556]    |
|0.0  |[-1015424.9999955161,-5.588170069463957,0.6676690508964003,-1.361404331763773]    |
|0.0  |[-1016276.9999890575,-15.329062861814776,-3.2858173413303966,-3.615304496714592]  |
|0.0  |[-1017022.9999951995,-6.1067873494330325,0.15795872609374673,-1.3613840782950968] |
|1.0  |[-1017121.9999798193,-25.234192348713805,0.5976006764167439,-0.7593487318928729]  |
|0.0  |[-1018098.9999913658,-8.525308769706054,6.7531407540177915,1.1189454999829327]    |
|0.0  |[-1018560.9999961505,-5.246642333184979,-0.3491622

In [39]:
print("--- Model Comparison ---")
print(f"Random Forest (Original Features) Precision: {precision_original}")
print(f"Random Forest (Original Features) Error Rate: {1 - precision_original}")
print(f"\nRandom Forest (PCA Features) Precision: {precision_pca}")
print(f"Random Forest (PCA Features) Error Rate: {1 - precision_pca}")

if precision_original > precision_pca:
    print("\nThe Random Forest model with original features performed better.")
elif precision_pca > precision_original:
    print("\nThe Random Forest model with PCA features performed better.")
else:
    print("\nBoth Random Forest models performed similarly.")

--- Model Comparison ---
Random Forest (Original Features) Precision: 0.9911025145067698
Random Forest (Original Features) Error Rate: 0.008897485493230217

Random Forest (PCA Features) Precision: 0.9729698197783303
Random Forest (PCA Features) Error Rate: 0.027030180221669697

The Random Forest model with original features performed better.
